# Pick coordinates from any image

Make sure you have an image of which you know the extents.

## Pick point on a map

One way to get it, is to load an image in GoogleEarth:

**In GoogleEarth app do**

add>image_overlay

**load the image and match the image with the GoogleEarth background**

Take the WGS84 coordinates of the window (location).

**Convert them to your reference system**
Put the convered ponts in an extent tuple or extent array:

extent=(xmin, xmax, ymin, ymax)

**Use ImagePicker to Load the image and start clicking your points**

pnts = ImagePicker(image_file_name)

pick the points with the mouse and finalley press enter to stop and cet all clicked points.

## Pick points in a graph

In this case cut out the graph so you know its extent.
Use this extent with the ImagePickler and pick the points from the figure.

You can also use a digitizer app on the internet, which will be much more advanced
and can get entire graphs based on their color.

e.g. https://plotdigitizer.com

which used to be free but now wants some money ($40,- (2026)).

Most famous sofware to do this:
webplotdigitizer https://apps.automeris.io/wpd4/ which is free

From the same person:
https://web.eecs.utk.edu/~dcostine/personal/PowerDeviceLib/DigiTest/index.html

20 minute instruction video
https://www.youtube.com/watch?v=YfomBJHU9fc





**Make sure the correct backend is used that allows interactive use of figure in a notebook**

With this backend, you only see the image after plt.show() and you must close it to continue.

plt.close('all') may work.

In [1]:
import matplotlib
matplotlib.use("QtAgg")

import matplotlib.pyplot as plt
print(matplotlib.get_backend())

QtAgg


**Imports**

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tools.etc import pickleto, picklefrom, descr

NOTEBOOK_NAME = "impage_picker.ipynb"
print(f"NOTEBOOK_NAME = '{NOTEBOOK_NAME}'")

matplotlib.use("QtAgg")
print(matplotlib.get_backend())

# --- project directory namespace
class Dirs:
    def __init__(self):
        self.rws = "/Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/"
        self.home = os.path.join(self.rws, "src")
        self.images = os.path.join(self.home, '../images')
        self.data = os.path.join(self.home, '../data')
        
dirs = Dirs()

print("Project directory namespace:")
descr(dirs)

NOTEBOOK_NAME = 'impage_picker.ipynb'
QtAgg
Project directory namespace:
rws                  /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/
home                 /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/src
images               /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/src/../images
data                 /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/src/../data


# Image Pickler

In [3]:
class ImagePicker:
    def __init__(self, image, extent):
        self.image = image
        self.extent = extent

    def pick_points(self, n=1, zoom=False, title="Click points"):

        plt.close('all')

        fig, ax = plt.subplots()

        ax.imshow(
            self.image,
            extent=self.extent,
            origin='upper'
        )

        ax.set_title(title + "\nClick points, ENTER to finish")

        if zoom:
            ax.axis('on')
        else:
            ax.axis('off')

        pts = []

        def onclick(event):
            if event.inaxes != ax:
                return

            if len(pts) >= n:
                return

            x, y = event.xdata, event.ydata
            pts.append((x, y))

            # immediate visual feedback
            ax.plot(x, y, 'r+', markersize=12, mew=2)

            fig.canvas.draw()

        def onkey(event):
            if event.key == 'enter':
                plt.close(fig)

        cid_click = fig.canvas.mpl_connect('button_press_event', onclick)
        cid_key = fig.canvas.mpl_connect('key_press_event', onkey)

        plt.show()

        # clean up connections
        fig.canvas.mpl_disconnect(cid_click)
        fig.canvas.mpl_disconnect(cid_key)

        return pts

# Example two images that were cutout from reports

In [4]:
image_wellen = os.path.join(dirs.images,
        "wellen_Beemster_22_fig3_1_128499_131432_473951_476274.png")
image_raaien = os.path.join(dirs.images,
        "raaien_fig2_1_systeemanalyse_Deltares_128127_130749_471773_475601.png")

for image in [image_wellen, image_raaien]:
    assert os.path.isfile(image), f"FileNotFound {image}"

fig, (ax1, ax2) = plt.subplots(1,2)

img_wellen = Image.open(image_wellen)
img_raaien = Image.open(image_raaien)
ax1.imshow(img_wellen)
ax2.imshow(img_raaien)
ax1.set_title("Wellen")
ax2.set_title("Raaien")

plt.show()

## Extent of the cutout of fig. 3.1 from Beemster et all to pick the wellen

In [3]:
from pyproj import Transformer

def convert_coords(lonlat, fr_crs='EPSG:4326', to_crs='EPSG:28992'):
    """Return points from fr_crs to to_crs coordinates system.

    Parameters
    ----------
    lonlat: input points ((lon,lat), ...) or ((x,y), ...))
        Input coordinates in from_crs system.
    fr_crs: str
        From coordinate reference system (Detault EPSG:4326 (WGS84))
    to_crs: str
        To coordinate reference system (Default EPSG:28992 (Amersfoort, Rijksdriehoeksmeting)

    Example: Geolocating cut-out of fig 3.1 in Beemster (2022)
    lonlat = np.array([[4.998927,  52.273840],
                    [5.042055,  52.253094]]
    )
    result: [128499., 131432., 473951., 476274.]
    
    >> convert_coords([[4.998927,  52.273840], [5.042055,  52.253094]])
    >> array([[128499., 476274.],
       [131432., 473951.]])
    """
    transformer = Transformer.from_crs(fr_crs, to_crs, always_xy=True)
    
    out = []
    for lon, lat in lonlat:
        x, y = transformer.transform(lon, lat)
        out.append((x,y))
        
    return np.array(out)

def convert_extent(extent, fr_crs='EPSG:4326', to_crs='EPSG:28992'):
    """Return extent in to_crs coordinate system."""  
    lonlat = np.array([(extent[ 0], extent[ 2]),
                       (extent[-3], extent[-1])])
    xy = convert_coords(lonlat, fr_crs=fr_crs, to_crs=to_crs)
    return xy.ravel()[[0, 2, 1, 3]]
    
    
# --- Verify forward and backward transformation
lonlat = np.array([[4.998927,  52.273840],
                    [5.042055,  52.253094]]
    )

print(f'\nInitial lonlat : {lonlat}')
    
xy = convert_coords([[4.998927,  52.273840], [5.042055,  52.253094]])
print(f'\nForward xy : {np.round(xy, 0)}')

latlon = convert_coords(xy, fr_crs='EPSG:28992', to_crs='EPSG:4326')
print(f"\nBackward lonlat: {lonlat}")

wgs_extent = lonlat.ravel()[[0, 2, 1, 3]]
print(f"\nInitial wgs_extent : {wgs_extent}")

rd_extent = convert_extent(wgs_extent)
print(f'\nForward rd_extent: {np.round(rd_extent, 0)}')

wgs_extent = convert_extent(rd_extent, fr_crs='EPSG:28992', to_crs='EPSG:4326')
print(f"\nBacktward wgs_extent: {wgs_extent}")


Initial lonlat : [[ 4.998927 52.27384 ]
 [ 5.042055 52.253094]]

Forward xy : [[128499. 476274.]
 [131432. 473951.]]

Backward lonlat: [[ 4.998927 52.27384 ]
 [ 5.042055 52.253094]]

Initial wgs_extent : [ 4.998927  5.042055 52.27384  52.253094]

Forward rd_extent: [128499. 131432. 476274. 473951.]

Backtward wgs_extent: [ 4.998927  5.042055 52.27384  52.253094]


In [6]:
# === Geolocating cut-out of fig.2.1 in Systeemanalyse Deltares
# --- NW SE (UL, LR) WGS84 (EPSG:4326) coordinates of image in GoogleEarth
lonlat = np.array([[4.993524,  52.267770],
                   [5.032213,  52.233491]]
)

# --- Resulting extent from geolocating image
extent_raaien = (128127, 130749, 471773, 475601)
convert_coords(lonlat)


array([[128126.92876311, 475600.68494673],
       [130749.20755831, 471773.12850052]])

In [7]:
# Other conversions of GE overlays in this project
[5.017117, 52.265786] # Hart ARK in dwsn Deltares
print(f"{'hart_dwarsdsn_Deltares_'}" + '_'.join([f"{p:.0f}" for p in convert_coords([[5.017117, 52.265786]]).ravel()]))

print("map_peilgeb_" + '_'.join([f"{p:.0f}" for p in convert_extent([4.957492, 5.062972, 52.210322, 52.302211])]))
print("map_dwarsdsn_Deltares_" + '_'.join([f"{p:.0f}" for p in convert_extent([4.947826, 5.093649 ,52.168706, 52.321159])]) + "__hart_129736_475371")
print("map_pb_tauw_N_" + '_'.join([f"{p:.0f}" for p in convert_extent([4.981701, 5.048669, 52.239599, 52.299198])]))
print("map_pb_tauw_Z_" + '_'.join([f"{p:.0f}" for p in convert_extent([4.967615, 5.033675, 52.186256, 52.245276])]))
   

hart_dwarsdsn_Deltares_129736_475371
map_peilgeb_125629_132885_469223_479409
map_dwarsdsn_Deltares_124941_134986_464597_481509__hart_129736_475371
map_pb_tauw_N_127302_131907_472471_479078
map_pb_tauw_Z_126306_130855_466541_473084


## Pick the wellen from the image

With ImagePicker passing filename and extent, the
image fires up and you can start clicking.
The output is the points as a list of (x,y) tuples
in the correct coordinate system.

In [8]:
if False:
    impckr = ImagePicker(img_wellen, extent=extent_wellen)
    pnts = impckr.pick_points(n=1000, zoom=True, title=os.path.basename(image_wellen))
    pnts = np.array(pnts).round()
    print("Clicked points")
    print(pnts)

**Don't forget to store the points, for instance, by pickling them.***

In [9]:
# pickleto(pnts, os.path.join(dirs.data, 'wellen.pkl'))

**Later on, the points can be retrieved by unpickling them**

In [10]:
fig, ax = plt.subplots()
ax.imshow(img_wellen, extent=extent_wellen, origin='upper')

pnts = np.array(picklefrom(os.path.join(dirs.data, 'wellen.pkl')))
for pnt in pnts:
    ax.plot(*pnt, 'ro')
    
plt.show()

NameError: name 'extent_wellen' is not defined

## Pickle raaien

In [ ]:
if False:
    impckr = ImagePicker(img_raaien, extent=extent_raaien)
    pnts = impckr.pick_points(n=1000, zoom=True, title=os.path.basename(image_wellen))
    pnts = np.array(pnts).round()    
    print("Clicked points")
    print(pnts)    


In [ ]:
# pickleto(pnts, os.path.join(dirs.data, 'raaien.pkl'))

In [ ]:
fig, ax = plt.subplots()
ax.imshow(img_raaien, extent=extent_raaien, origin='upper')

pnts = np.array(picklefrom(os.path.join(dirs.data, 'raaien.pkl')))
for pnt in pnts:
    ax.plot(*pnt, 'ro')
    
plt.show()


## Heart line ARK

In [ ]:
if False:
    impckr = ImagePicker(img_raaien, extent=extent_raaien)
    pnts = impckr.pick_points(n=1000, zoom=True, title=os.path.basename(image_wellen))
    pnts = np.array(pnts).round()    
    print("Clicked points")
    print(pnts)    

In [ ]:
ARK_heart_line = np.array([
            [129779., 475561.],
            [129139., 471848.]
])
            
ARK_center_line_GE = np.array([
            [129810.0, 475695.0],
            [129006.0, 471010.0]
             ])

In [13]:
hartlijn_ARK_GE = np.array([
 [4.994432844320089,52.18058173605287],
 [4.995699526423916,52.18598913196669],
 [4.996940095378958,52.19067749758187],
 [4.998707417645091,52.19705663665697],
 [5.000288575411728,52.20291016621028],
 [5.001425914402855,52.20706580467734],
 [5.002605495791402,52.21127604079135],
 [5.004529633983259,52.21862801758714],
 [5.005447899806099,52.22195612356732],
 [5.007010123801905,52.22755675784340],
 [5.008530042812655,52.23316919352392],
 [5.010534335011824,52.24046322905308],
 [5.012147906796072,52.24642673961877],
 [5.013760047089773,52.25229784945274],
 [5.015902943528287,52.26013929956371],
 [5.019490544366574,52.27361416925556],
 [5.019944838628403,52.27488034151383],
 [5.021661263864006,52.28051264180155],
 [5.021715456486316,52.28310192838920],
 [5.021700503175445,52.28878941926938],
 [5.021641639425192,52.29126077391599],
 [5.021562823076044,52.29637337850166],
 [5.021545056890262,52.30103061184899],
 [5.021436485881079,52.30203536137964],
 [5.021429249174805,52.30285095607465]
 ])

np.round(convert_coords(hartlijn_ARK_GE), 0)[::-1]

array([[130051., 479494.],
       [130051., 479403.],
       [130058., 479291.],
       [130057., 478773.],
       [130059., 478204.],
       [130062., 477929.],
       [130060., 477296.],
       [130055., 477008.],
       [129934., 476382.],
       [129903., 476242.],
       [129650., 474744.],
       [129499., 473872.],
       [129386., 473219.],
       [129272., 472556.],
       [129131., 471746.],
       [129024., 471122.],
       [128914., 470499.],
       [128849., 470129.],
       [128714., 469312.],
       [128630., 468844.],
       [128550., 468382.],
       [128439., 467731.],
       [128314., 467022.],
       [128226., 466501.],
       [128136., 465900.]])

In [5]:
peilbuizen_in_en_naast_kanaal = np.array([
[5.016715, 52.265706],
[5.018398, 52.265818],
[5.018015, 52.265818],
[5.015582, 52.261546],
[5.016921, 52.261555],
[5.017134, 52.235566],
[5.008487, 52.235173],
[5.009696, 52.235213],
[5.010163, 52.235229],
[5.001770, 52.211509]   
])

np.round(convert_coords(peilbuizen_in_en_naast_kanaal))

array([[129709., 475363.],
       [129824., 475375.],
       [129798., 475375.],
       [129629., 474900.],
       [129720., 474901.],
       [129720., 472009.],
       [129129., 471968.],
       [129212., 471972.],
       [129244., 471974.],
       [128657., 469338.]])

## Unpickle the stored data and plot them with the the two heartlines and the image

It shows that both heartlines are almost identical even though that of Google Earth directly should be more accurate.

In [ ]:
wellen = picklefrom(os.path.join(dirs.data, "wellen.pkl"))
raaien = picklefrom(os.path.join(dirs.data, "raaien.pkl"))

fig, ax = plt.subplots()
ax.plot(*ARK_heart_line.T, label="heart_line ARK (Image)")
ax.plot(*ARK_center_line_GE.T, label="center_line_ARK (GE)")
ax.plot(*wellen.T, 'bo', label="wellen")
ax.plot(*raaien.T, 'ro', label='raaien')

if False:
    image = "wellen_Beemster_22_fig3_1_128499_131432_473951_476274.png"
    extent = extent_wellen
else:
    image = "raaien_fig2_1_systeemanalyse_Deltares_128127_130749_471773_475601.png"
    extent = extent_raaien

img = Image.open(os.path.join(dirs.images, image))
#ax.imshow(img, extent=extent_wellen, origin='upper')
ax.imshow(img, extent=extent, origin='upper')
ax.legend()
plt.show()

In [ ]:
def point_line_distance(P, A, B):
    """
    Distance from points P to infinite line through A and B.

    Parameters
    ----------
    P : (..., 2) array_like
        One or more points.
    A, B : (2,) array_like
        Two points defining the line.

    Returns
    -------
    d : ndarray
        Distances.
    """

    P = np.asarray(P)
    A = np.asarray(A)
    B = np.asarray(B)

    AB = B - A
    AP = P - A

    cross = AB[0] * AP[..., 1] - AB[1] * AP[..., 0]

    return np.round(np.abs(cross) / np.linalg.norm(AB), 2), np.sign(cross)

    
# --- Afstand wellen tot ARK center line
A, B = ARK_center_line_GE
wellen = np.array(picklefrom(os.path.join(dirs.data, 'wellen.pkl')))
raaien = np.array(picklefrom(os.path.join(dirs.data, 'raaien.pkl')))

d_wellen, sign_wellen = point_line_distance(wellen, A, B)
print("\nDistance wellen to center line of ARK [m]")
print(d_wellen)
d_raaien, sign_raaien = point_line_distance(raaien, A, B)
print("\nDistance obs. wells to center line of ARK [m]")
print(d_raaien)

fig, ax = plt.subplots()

ax.plot(d_wellen * sign_wellen, np.zeros_like(d_wellen), 'bo', label='wellen')
ax.plot(d_raaien * sign_raaien, np.zeros_like(d_raaien), 'ro', label='buizen')

ax.set_title("Aftand wellen en peilbuizen tot hartlijen of ARK")
ax.grid()
ax.legend()

plt.show()
